In [ ]:
# [0 · Imports & configuration]
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import kruskal

sys.path.insert(0, '/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data')
from nalm_utils import (
    display_name, get_marker_cols, get_pairwise_cols,
    build_mean_matrix, compute_ward_linkage, select_top_partners,
    run_spatial_comparison, draw_force_net,
    plot_da_bars, sig_label,
)

sc.set_figure_params(dpi=100, frameon=False)
plt.rcParams['figure.max_open_warning'] = 0

CACHE_DIR  = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache')
ADATA_PATH = CACHE_DIR / 'adata_cytovi_annotated_compat.h5ad'

MARKERS = [
    'CD18', 'CD11a', 'CD50', 'CD29', 'CD49D',   # pSMAC adhesion
    'GPR56', 'CX3CR1', 'CD226', 'CD154',          # effector / activation
]

SYS_HH = 'healthy B + healthy T'
SYS_NH = 'NALM-6 + healthy T'
SYS_PP = 'patient B + patient T'
SYS_NP = 'NALM-6 + patient T'
ALL_SYSTEMS = [SYS_HH, SYS_NH, SYS_PP, SYS_NP]

SYSTEM_PALETTE = {
    SYS_HH: '#4878d0',
    SYS_NH: '#ee854a',
    SYS_PP: '#6acc65',
    SYS_NP: '#d65f5f',
}
SYS_SHORT = {
    SYS_HH: 'HB+HT',
    SYS_NH: 'N6+HT',
    SYS_PP: 'PB+PT',
    SYS_NP: 'N6+PT',
}

In [ ]:
# [1 · Data loading & CD8 subsets]
adata = sc.read_h5ad(ADATA_PATH)
print(f'Loaded {adata.n_obs:,} cells x {adata.n_vars} markers')

present = [m for m in MARKERS if m in adata.var_names]
missing = [m for m in MARKERS if m not in adata.var_names]
print(f'Markers present ({len(present)}): {present}')
if missing:
    print(f'Missing: {missing}')
MARKERS = present

# All CD8 cells across all 4 systems
adata_cd8 = adata[
    (adata.obs['cell_type_annot'] == 'CD8') &
    (adata.obs['cell_system'].isin(ALL_SYSTEMS))
].copy()
adata_cd8.obs['time_cond'] = (
    adata_cd8.obs['time'].astype(str) + ' ' + adata_cd8.obs['condition'].astype(str)
)

print(f'\nCD8 cells across all 4 systems: {adata_cd8.n_obs}')
pd.crosstab(adata_cd8.obs['cell_system'], [adata_cd8.obs['condition'], adata_cd8.obs['time']])

In [ ]:
# [1b · Pick a doublet, load its 3D graph layout, mark CD3e + CD19]
import plotly.graph_objects as go
from pixelator import read_pna

RESULTS_DIR = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/results')

# Prefer a Blinatumomab + NALM-6 + 48h doublet (strongest synapse signal); fall back to any.
dbl = adata.obs[adata.obs['cell_type_annot'] == 'Doublets']
pref = dbl[(dbl['condition'] == 'Blinatumomab')
           & (dbl['time'] == '48h')
           & (dbl['cell_system'].isin(['NALM-6 + healthy T', ]))]
picked = (pref if len(pref) else dbl).iloc[3]
component_id = picked.name
sample_id    = picked['sample']
print(f'Doublet {component_id}  sample={sample_id}  '
      f'{picked["condition"]}/{picked["time"]}  {picked["cell_system"]}')

# Load the .pxl for that sample and pull only this component's precomputed layout.
pxl_path = RESULTS_DIR / sample_id / 'layout' / 'layout' / f'{sample_id}.layout.pxl'
pg = read_pna([pxl_path])
layout_df = (
    pg.filter(components=[component_id])
      .precomputed_layouts()
      .to_df()
)
print(f'Layout: {layout_df.shape}  columns sample: {list(layout_df.columns[:10])} ...')

# CD3e = T marker, CD19 = B marker; the wide-format layout has one column per marker.
MARK_COLS  = {'CD3e': 'crimson', 'CD19': 'royalblue'}
missing = [m for m in MARK_COLS if m not in layout_df.columns]
if missing:
    raise KeyError(f'Markers missing from layout columns: {missing}')

fig = go.Figure()
# All vertices as a light grey cloud for the cell surface.
fig.add_trace(go.Scatter3d(
    x=layout_df['x'], y=layout_df['y'], z=layout_df['z'],
    mode='markers',
    marker=dict(size=1.4, color='#999999', opacity=0.55),
    name='surface',
    showlegend=True,
))
# Highlight vertices positive for each synapse marker.
for mk, color in MARK_COLS.items():
    pos = layout_df[layout_df[mk] > 0]
    fig.add_trace(go.Scatter3d(
        x=pos['x'], y=pos['y'], z=pos['z'],
        mode='markers',
        marker=dict(size=3.0, color=color, opacity=0.95),
        name=f'{mk} (n={len(pos)})',
    ))

fig.update_layout(
    title=f'Doublet {component_id} — {sample_id} '
          f'({picked["condition"]}/{picked["time"]}, {picked["cell_system"]})',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z',
               aspectmode='data'),
    width=800, height=700, legend=dict(itemsizing='constant'),
)
html_path = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/factor_analysis/results_semantic') / f'doublet_{component_id}_{sample_id}.html'
html_path.parent.mkdir(parents=True, exist_ok=True)
fig.write_html(html_path, include_plotlyjs='cdn', full_html=True)
print(f'Saved interactive HTML -> {html_path}')
fig.show()


In [ ]:
adata.obs.cell_type_annot.value_counts()

In [ ]:
doublet=adata[adata.obs.cell_type_annot=='Doublets'].obs_names[0]
doublet


## Abundance — all 4 cell systems

In [ ]:
# [2 · Abundance violins — one figure per marker, all 4 systems across conditions]
X = pd.DataFrame(
    np.array(adata_cd8.layers['arcsinh'], dtype=np.float32),
    index=adata_cd8.obs_names, columns=adata_cd8.var_names,
)

TC_ORDER = [t for t in ['6h Mock', '6h Blinatumomab', '48h Mock', '48h Blinatumomab']
            if t in adata_cd8.obs['time_cond'].unique()]
TC_LABELS = {
    '6h Mock': '6h\nMock', '6h Blinatumomab': '6h\nBlina',
    '48h Mock': '48h\nMock', '48h Blinatumomab': '48h\nBlina',
}

sys_order_short = [SYS_SHORT[s] for s in ALL_SYSTEMS]
palette_short   = {SYS_SHORT[s]: c for s, c in SYSTEM_PALETTE.items()}
tc_order_short  = [TC_LABELS[t] for t in TC_ORDER]

for marker in MARKERS:
    df_v = pd.DataFrame({
        'expression': X[marker].values,
        'Condition':  adata_cd8.obs['time_cond'].map(TC_LABELS).values,
        'System':     adata_cd8.obs['cell_system'].map(SYS_SHORT).values,
    })

    # KW significance per system across conditions
    kw_parts = []
    for sys_full, sys_short in SYS_SHORT.items():
        sys_mask = (adata_cd8.obs['cell_system'] == sys_full).values
        arrays = [
            X.loc[sys_mask & (adata_cd8.obs['time_cond'] == tc).values, marker].values
            for tc in TC_ORDER
        ]
        arrays = [a for a in arrays if len(a) > 1]
        if len(arrays) > 1:
            _, pval = kruskal(*arrays)
            kw_parts.append(f'{sys_short}: {sig_label(pval)}')

    fig, ax = plt.subplots(figsize=(14, 5))
    sns.violinplot(
        data=df_v, x='Condition', y='expression', hue='System',
        order=tc_order_short, hue_order=sys_order_short,
        palette=palette_short, cut=0, inner='quartile',
        density_norm='width', ax=ax,
    )
    ax.set_title(
        f'{display_name(marker)}    KW across conditions — ' + '  |  '.join(kw_parts),
        fontsize=12, fontweight='bold',
    )
    ax.set_xlabel('')
    ax.set_ylabel('arcsinh')
    ax.legend(title='System', fontsize=9, loc='upper right', ncol=2)
    plt.tight_layout()
    plt.show()

In [ ]:
# [3 · DA bars: NALM-6+HT vs NALM-6+PT — 6h Blinatumomab]
# Mirror of nalm_analysis cell [3]: plot_da_bars(adata_sub, cell_types, sys_a, sys_b)
adata_nalm_cd8 = adata_cd8[
    adata_cd8.obs['cell_system'].isin([SYS_NH, SYS_NP])
].copy()

adata_6h_blina = adata_nalm_cd8[
    (adata_nalm_cd8.obs['time'] == '6h') &
    (adata_nalm_cd8.obs['condition'] == 'Blinatumomab')
][:, MARKERS].copy()

plot_da_bars(adata_6h_blina, ['CD8'], SYS_NH, SYS_NP)

In [ ]:
# [4 · DA bars: NALM-6+HT vs NALM-6+PT — 48h Blinatumomab]
adata_48h_blina = adata_nalm_cd8[
    (adata_nalm_cd8.obs['time'] == '48h') &
    (adata_nalm_cd8.obs['condition'] == 'Blinatumomab')
][:, MARKERS].copy()

plot_da_bars(adata_48h_blina, ['CD8'], SYS_NH, SYS_NP)

## Proximity — Healthy T vs Patient T co-cultured with NALM-6

In [ ]:
# [5 · Build spatial subsets — CD8, NALM-6+HT and NALM-6+PT, all needed conditions]
# Mirror of nalm_analysis cell [13]
def _sp_subset(adata_full, time_val, cond_val, system_val, label=''):
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['condition'] == cond_val) &
        (adata_full.obs['cell_type_annot'] == 'CD8') &
        (adata_full.obs['cell_system'] == system_val)
    )
    sp = adata_full[mask].obsm['spatial_asinh5']
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=adata_full.obs_names[mask])
    tag = label or f'{time_val} {cond_val[:5]} {system_val[:6]}'
    print(f'  {tag:42s} → {sp.shape[0]:4d} cells')
    return sp

print('Building CD8 spatial subsets:')
sp_6h_mock_NH  = _sp_subset(adata, '6h',  'Mock',         SYS_NH, 'NALM-6 + Healthy T   6h  Mock')
sp_6h_mock_NP  = _sp_subset(adata, '6h',  'Mock',         SYS_NP, 'NALM-6 + Patient T   6h  Mock')
sp_6h_blina_NH = _sp_subset(adata, '6h',  'Blinatumomab', SYS_NH, 'NALM-6 + Healthy T   6h  Blina')
sp_6h_blina_NP = _sp_subset(adata, '6h',  'Blinatumomab', SYS_NP, 'NALM-6 + Patient T   6h  Blina')
sp_48h_blina_NH = _sp_subset(adata, '48h', 'Blinatumomab', SYS_NH, 'NALM-6 + Healthy T  48h  Blina')
sp_48h_blina_NP = _sp_subset(adata, '48h', 'Blinatumomab', SYS_NP, 'NALM-6 + Patient T  48h  Blina')

all_sp_cols = sp_6h_mock_NH.columns
print(f'\nTotal colocalization pairs: {len(all_sp_cols)}')

In [ ]:
# [6 · CX3CR1 differential colocalization — 4 focused comparisons]
# Only differential plot + print (no top-partners); mirror of nalm_analysis plot_diff_coloc calls
from nalm_utils import plot_diff_coloc

MARKER = 'CX3CR1'
cx3cr1_cols = [c for c in all_sp_cols if MARKER in c.split('/')]

comparisons = [
    (sp_6h_mock_NP,   sp_6h_mock_NH,   'N6+PT 6h Mock',   'N6+HT 6h Mock'),
    (sp_6h_blina_NP,  sp_6h_blina_NH,  'N6+PT 6h Blina',  'N6+HT 6h Blina'),
    (sp_48h_blina_NP, sp_48h_blina_NH, 'N6+PT 48h Blina', 'N6+HT 48h Blina'),
    (sp_48h_blina_NH, sp_6h_blina_NH,  'N6+HT 48h Blina', 'N6+HT 6h Blina'),
]

for sp_a, sp_b, label_a, label_b in comparisons:
    plot_diff_coloc(
        sp_a, sp_b, [MARKER], all_sp_cols,
        sys_a_label=label_a, sys_b_label=label_b,
    )

## Neighborhood heatmaps — all 9 markers

In [ ]:
# [10 · Shared neighborhood heatmaps + networks — 4 conditions]
# Mirror of nalm_analysis cell [25]
top_partners_all = set(MARKERS)
for sp in [sp_6h_NH, sp_48h_NH, sp_6h_NP, sp_48h_NP]:
    for m in MARKERS:
        top_partners_all.update(select_top_partners(sp, m, all_sp_cols, n=10))

combined_markers = sorted(top_partners_all)
pair_cols = get_pairwise_cols(all_sp_cols, top_partners_all)
print(f'Combined marker set ({len(combined_markers)}): {[display_name(m) for m in combined_markers]}')

mat_6h_NH  = build_mean_matrix(sp_6h_NH,  pair_cols, combined_markers)
mat_48h_NH = build_mean_matrix(sp_48h_NH, pair_cols, combined_markers)
mat_6h_NP  = build_mean_matrix(sp_6h_NP,  pair_cols, combined_markers)
mat_48h_NP = build_mean_matrix(sp_48h_NP, pair_cols, combined_markers)

mat_avg     = (mat_6h_NH + mat_48h_NH + mat_6h_NP + mat_48h_NP) / 4
row_linkage = compute_ward_linkage(mat_avg)

vmax    = max(m.abs().max().max() for m in [mat_6h_NH, mat_48h_NH, mat_6h_NP, mat_48h_NP])
n_m     = len(combined_markers)
fig_sz  = max(10, n_m * 0.22)
tick_fs = max(5, min(8, 200 // n_m))

oi_set   = set(MARKERS)
node_cmap = {m: ('#e41a1c' if m in oi_set else '#555555') for m in combined_markers}

for mat, label in [
    (mat_6h_NH,  'NALM-6 + Healthy T  —  6h Blina'),
    (mat_48h_NH, 'NALM-6 + Healthy T  —  48h Blina'),
    (mat_6h_NP,  'NALM-6 + Patient T  —  6h Blina'),
    (mat_48h_NP, 'NALM-6 + Patient T  —  48h Blina'),
]:
    dn    = {m: display_name(m) for m in mat.index}
    mat_d = mat.rename(index=dn, columns=dn)

    g = sns.clustermap(
        mat_d, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
        row_linkage=row_linkage, col_linkage=row_linkage,
        figsize=(fig_sz, fig_sz), linewidths=0,
        xticklabels=True, yticklabels=True,
        cbar_kws={'shrink': 0.4, 'label': 'mean z-score'},
        dendrogram_ratio=0.08, cbar_pos=(0.02, 0.82, 0.03, 0.15),
    )
    g.ax_heatmap.tick_params(axis='both', labelsize=tick_fs)
    g.fig.suptitle(f'pSMAC/effector neighborhood — {label}', fontsize=12, y=1.01)
    plt.show()

    fig_net, ax_net = plt.subplots(figsize=(10, 10))
    draw_force_net(
        mat, f'Neighborhood network — {label}',
        ax=ax_net, top_n=80, highlight_node='CD11a', node_color_map=node_cmap,
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# [11 · Difference heatmaps + networks — 4 key contrasts]
# Mirror of nalm_analysis cell [26]
contrasts = [
    (mat_6h_NH  - mat_6h_NP,   'Healthy T − Patient T  (6h Blina)'),
    (mat_48h_NH - mat_48h_NP,  'Healthy T − Patient T  (48h Blina)'),
    (mat_48h_NH - mat_6h_NH,   'NALM-6+HT:  48h − 6h Blina'),
    (mat_48h_NP - mat_6h_NP,   'NALM-6+PT:  48h − 6h Blina'),
]

for mat_diff, title in contrasts:
    vmax_d = mat_diff.abs().max().max()
    dn     = {m: display_name(m) for m in mat_diff.index}
    mat_d  = mat_diff.rename(index=dn, columns=dn)

    g = sns.clustermap(
        mat_d, cmap='coolwarm', center=0, vmin=-vmax_d, vmax=vmax_d,
        row_linkage=row_linkage, col_linkage=row_linkage,
        figsize=(fig_sz, fig_sz), linewidths=0,
        xticklabels=True, yticklabels=True,
        cbar_kws={'shrink': 0.4, 'label': 'diff (mean z-score)'},
        dendrogram_ratio=0.08, cbar_pos=(0.02, 0.82, 0.03, 0.15),
    )
    g.ax_heatmap.tick_params(axis='both', labelsize=tick_fs)
    g.fig.suptitle(f'Neighborhood difference — {title}', fontsize=12, y=1.01)
    plt.show()

    fig_net, ax_net = plt.subplots(figsize=(10, 10))
    draw_force_net(
        mat_diff, f'Difference network — {title}',
        ax=ax_net, top_n=60, highlight_node='CD11a', node_color_map=node_cmap,
    )
    plt.tight_layout()
    plt.show()